# 🫀 CardioAI - Fase 4: CNN para Classificação de ECG

**FIAP - Inteligência Artificial Aplicada à Cardiologia**

---

## Objetivo

Desenvolver um sistema de classificação de imagens de ECG usando:
- CNN do zero (scratch)
- Transfer Learning (MobileNetV2, VGG16, ResNet50)
- Avaliação com métricas completas
- Análise de viés e governança em IA médica

## ⚠️ AVISO IMPORTANTE

Este é um **protótipo acadêmico** desenvolvido para fins educacionais.
**NÃO deve ser usado para diagnóstico médico real.**
Sempre consulte profissionais de saúde qualificados.

---

## 1. Configuração do Ambiente

In [ ]:
# Instalar dependências (se necessário)
!pip install -q tensorflow matplotlib seaborn scikit-learn pillow

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
import warnings

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

warnings.filterwarnings('ignore')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponível: {tf.config.list_physical_devices('GPU')}")

## 2. Carregamento e Exploração do Dataset

### 2.1 Upload do Dataset

**Opção 1: Google Drive**
```python
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/ecg_images'
```

**Opção 2: Kaggle API**
```python
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d shayanfazeli/heartbeat
!unzip -q heartbeat.zip -d ecg_images
```

**Opção 3: Upload Manual**

In [ ]:
# Configurar caminho do dataset
# Ajuste conforme sua estrutura
DATA_DIR = '/content/ecg_images'  # Colab
# DATA_DIR = '../../data/raw/ecg_images'  # Local

# Verificar estrutura
if os.path.exists(DATA_DIR):
    print("✓ Dataset encontrado!")
    print("\nClasses disponíveis:")
    classes = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
    for cls in classes:
        count = len(os.listdir(os.path.join(DATA_DIR, cls)))
        print(f"  - {cls}: {count} imagens")
else:
    print("⚠ Dataset não encontrado. Faça o upload primeiro.")

### 2.2 Visualização de Amostras

In [ ]:
# Visualizar amostras de cada classe
def plot_samples(data_dir, classes, samples_per_class=3):
    fig, axes = plt.subplots(len(classes), samples_per_class, figsize=(12, len(classes)*3))
    
    for i, cls in enumerate(classes):
        class_dir = os.path.join(data_dir, cls)
        images = os.listdir(class_dir)[:samples_per_class]
        
        for j, img_name in enumerate(images):
            img_path = os.path.join(class_dir, img_name)
            img = Image.open(img_path)
            
            ax = axes[i, j] if len(classes) > 1 else axes[j]
            ax.imshow(img, cmap='gray')
            ax.axis('off')
            if j == 0:
                ax.set_title(f'{cls}', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

if os.path.exists(DATA_DIR):
    plot_samples(DATA_DIR, classes)

### 2.3 Análise de Distribuição de Classes (IR ALÉM 1)

In [ ]:
# Analisar distribuição de classes
class_counts = {}
for cls in classes:
    class_dir = os.path.join(DATA_DIR, cls)
    class_counts[cls] = len(os.listdir(class_dir))

# Criar DataFrame
df_dist = pd.DataFrame(list(class_counts.items()), columns=['Classe', 'Quantidade'])
df_dist['Percentual'] = (df_dist['Quantidade'] / df_dist['Quantidade'].sum() * 100).round(2)

print("\n📊 Distribuição de Classes:")
print(df_dist.to_string(index=False))

# Visualizar distribuição
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
ax1.bar(df_dist['Classe'], df_dist['Quantidade'], color='steelblue')
ax1.set_xlabel('Classe', fontsize=12)
ax1.set_ylabel('Quantidade de Imagens', fontsize=12)
ax1.set_title('Distribuição de Classes', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Gráfico de pizza
ax2.pie(df_dist['Quantidade'], labels=df_dist['Classe'], autopct='%1.1f%%', startangle=90)
ax2.set_title('Proporção de Classes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Análise de desbalanceamento
max_count = df_dist['Quantidade'].max()
min_count = df_dist['Quantidade'].min()
imbalance_ratio = max_count / min_count

print(f"\n⚠️ Análise de Desbalanceamento:")
print(f"  - Classe mais frequente: {df_dist.loc[df_dist['Quantidade'].idxmax(), 'Classe']} ({max_count} imagens)")
print(f"  - Classe menos frequente: {df_dist.loc[df_dist['Quantidade'].idxmin(), 'Classe']} ({min_count} imagens)")
print(f"  - Razão de desbalanceamento: {imbalance_ratio:.2f}x")

if imbalance_ratio > 2:
    print("\n🚨 ALERTA: Dataset significativamente desbalanceado!")
    print("   Considere técnicas de balanceamento ou ajuste de pesos nas classes.")

## 3. Pré-processamento e Data Augmentation

In [ ]:
# Configurações
IMG_SIZE = (256, 256)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
TEST_SPLIT = 0.1
SEED = 42

# Definir número de classes
NUM_CLASSES = len(classes)
print(f"Número de classes: {NUM_CLASSES}")
print(f"Classes: {classes}")

In [ ]:
# Data Augmentation para treino
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=VALIDATION_SPLIT + TEST_SPLIT
)

# Apenas normalização para validação e teste
val_test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT + TEST_SPLIT
)

# Criar geradores
train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='training',
    seed=SEED
)

val_generator = val_test_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='validation',
    seed=SEED,
    shuffle=False
)

print(f"\n✓ Datasets criados:")
print(f"  - Treino: {train_generator.samples} imagens")
print(f"  - Validação: {val_generator.samples} imagens")

## 4. Modelo CNN do Zero (Scratch)

In [ ]:
def build_cnn_scratch(input_shape, num_classes):
    """
    Constrói uma CNN simples do zero.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Bloco 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        # Bloco 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.2),
        
        # Bloco 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Classificador
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='CNN_Scratch')
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Criar modelo
model_scratch = build_cnn_scratch((*IMG_SIZE, 3), NUM_CLASSES)
model_scratch.summary()

In [ ]:
# Treinar modelo CNN do zero
EPOCHS = 20

history_scratch = model_scratch.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    verbose=1
)

print("\n✓ Treinamento da CNN do zero concluído!")

## 5. Transfer Learning com MobileNetV2

In [ ]:
def build_transfer_model(input_shape, num_classes, base_model_name='mobilenetv2'):
    """
    Constrói modelo com Transfer Learning.
    """
    # Carregar modelo base pré-treinado
    if base_model_name == 'mobilenetv2':
        base_model = applications.MobileNetV2(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    elif base_model_name == 'vgg16':
        base_model = applications.VGG16(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    elif base_model_name == 'resnet50':
        base_model = applications.ResNet50(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    
    # Congelar base convolucional
    base_model.trainable = False
    
    # Construir modelo completo
    model = models.Sequential([
        layers.Input(shape=input_shape),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name=f'Transfer_{base_model_name.upper()}')
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Criar modelo com Transfer Learning
model_transfer = build_transfer_model((*IMG_SIZE, 3), NUM_CLASSES, 'mobilenetv2')
model_transfer.summary()

In [ ]:
# Treinar modelo com Transfer Learning
history_transfer = model_transfer.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    verbose=1
)

print("\n✓ Treinamento com Transfer Learning concluído!")

## 6. Avaliação e Métricas

In [ ]:
# Plotar histórico de treinamento
def plot_training_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Treino', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Validação', linewidth=2)
    ax1.set_xlabel('Época', fontsize=12)
    ax1.set_ylabel('Accuracy', fontsize=12)
    ax1.set_title(f'{title} - Accuracy', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Loss
    ax2.plot(history.history['loss'], label='Treino', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Validação', linewidth=2)
    ax2.set_xlabel('Época', fontsize=12)
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.set_title(f'{title} - Loss', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plotar históricos
plot_training_history(history_scratch, 'CNN do Zero')
plot_training_history(history_transfer, 'Transfer Learning')

In [ ]:
# Avaliar modelos
def evaluate_model(model, generator, model_name):
    print(f"\n{'='*60}")
    print(f"Avaliação: {model_name}")
    print(f"{'='*60}")
    
    # Predições
    y_pred_prob = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = generator.classes
    
    # Métricas
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    
    print(f"\n📊 Métricas Gerais:")
    print(f"  - Accuracy:  {accuracy:.4f}")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall:    {recall:.4f}")
    print(f"  - F1-Score:  {f1:.4f}")
    
    # Relatório por classe
    print(f"\n📋 Relatório por Classe:")
    print(classification_report(y_true, y_pred, target_names=classes, digits=4))
    
    # Matriz de confusão
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap='Blues', xticks_rotation=45)
    plt.title(f'Matriz de Confusão - {model_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_pred_prob': y_pred_prob
    }

# Avaliar ambos os modelos
results_scratch = evaluate_model(model_scratch, val_generator, 'CNN do Zero')
results_transfer = evaluate_model(model_transfer, val_generator, 'Transfer Learning (MobileNetV2)')

In [ ]:
# Comparar modelos
comparison = pd.DataFrame({
    'Modelo': ['CNN do Zero', 'Transfer Learning'],
    'Accuracy': [results_scratch['accuracy'], results_transfer['accuracy']],
    'Precision': [results_scratch['precision'], results_transfer['precision']],
    'Recall': [results_scratch['recall'], results_transfer['recall']],
    'F1-Score': [results_scratch['f1'], results_transfer['f1']]
})

print("\n🏆 Comparação de Modelos:")
print(comparison.to_string(index=False))

# Visualizar comparação
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison['Modelo']))
width = 0.2

ax.bar(x - 1.5*width, comparison['Accuracy'], width, label='Accuracy', color='steelblue')
ax.bar(x - 0.5*width, comparison['Precision'], width, label='Precision', color='orange')
ax.bar(x + 0.5*width, comparison['Recall'], width, label='Recall', color='green')
ax.bar(x + 1.5*width, comparison['F1-Score'], width, label='F1-Score', color='red')

ax.set_xlabel('Modelo', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparação de Métricas entre Modelos', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Modelo'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Inferência em Nova Imagem

Teste o modelo com uma imagem enviada pelo usuário.

In [ ]:
# Upload de imagem para teste
from google.colab import files
from IPython.display import display

print("📤 Faça upload de uma imagem de ECG para classificação:")
uploaded = files.upload()

if uploaded:
    # Pegar primeira imagem
    image_path = list(uploaded.keys())[0]
    
    # Carregar e pré-processar
    img = Image.open(image_path).convert('RGB')
    img_resized = img.resize(IMG_SIZE)
    img_array = np.array(img_resized) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Fazer predição com ambos os modelos
    pred_scratch = model_scratch.predict(img_array, verbose=0)
    pred_transfer = model_transfer.predict(img_array, verbose=0)
    
    # Resultados
    class_scratch = classes[np.argmax(pred_scratch[0])]
    conf_scratch = np.max(pred_scratch[0]) * 100
    
    class_transfer = classes[np.argmax(pred_transfer[0])]
    conf_transfer = np.max(pred_transfer[0]) * 100
    
    # Visualizar
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    
    # Imagem original
    ax1.imshow(img)
    ax1.axis('off')
    ax1.set_title('Imagem Original', fontsize=14, fontweight='bold')
    
    # Predição CNN do Zero
    ax2.barh(classes, pred_scratch[0] * 100, color='steelblue')
    ax2.set_xlabel('Confiança (%)', fontsize=12)
    ax2.set_title(f'CNN do Zero\n{class_scratch} ({conf_scratch:.1f}%)', 
                  fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='x')
    
    # Predição Transfer Learning
    ax3.barh(classes, pred_transfer[0] * 100, color='orange')
    ax3.set_xlabel('Confiança (%)', fontsize=12)
    ax3.set_title(f'Transfer Learning\n{class_transfer} ({conf_transfer:.1f}%)', 
                  fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print("\n⚠️ LEMBRE-SE: Este é um resultado acadêmico e NÃO deve ser usado para diagnóstico médico real!")

## 8. IR ALÉM 1: Análise de Ética e Governança

### 8.1 Riscos Identificados

1. **Desbalanceamento de Classes**: Pode levar a viés nas predições
2. **Falsos Positivos**: Podem causar ansiedade desnecessária
3. **Falsos Negativos**: Podem atrasar diagnósticos reais
4. **Viés de Dataset**: Limitado a padrões específicos
5. **Falta de Explicabilidade**: Dificulta confiança clínica

In [ ]:
# Análise de erros por classe
def analyze_errors(y_true, y_pred, classes):
    print("\n🔍 Análise de Erros por Classe:\n")
    
    for i, cls in enumerate(classes):
        # Índices da classe
        class_indices = np.where(y_true == i)[0]
        
        if len(class_indices) == 0:
            continue
        
        # Predições para esta classe
        class_predictions = y_pred[class_indices]
        
        # Calcular erros
        correct = np.sum(class_predictions == i)
        total = len(class_indices)
        errors = total - correct
        error_rate = (errors / total) * 100
        
        print(f"{cls}:")
        print(f"  - Total: {total}")
        print(f"  - Corretos: {correct}")
        print(f"  - Erros: {errors}")
        print(f"  - Taxa de erro: {error_rate:.2f}%")
        
        # Identificar confusões
        if errors > 0:
            wrong_predictions = class_predictions[class_predictions != i]
            unique, counts = np.unique(wrong_predictions, return_counts=True)
            print(f"  - Confundido com:")
            for pred_class, count in zip(unique, counts):
                print(f"    • {classes[pred_class]}: {count} vezes")
        print()

# Analisar erros do melhor modelo
best_results = results_transfer if results_transfer['accuracy'] > results_scratch['accuracy'] else results_scratch
analyze_errors(best_results['y_true'], best_results['y_pred'], classes)

### 8.2 Estratégias de Mitigação

1. **Balanceamento de Classes**
   - Usar class weights
   - Aplicar oversampling/undersampling
   - Gerar dados sintéticos (SMOTE)

2. **Validação Externa**
   - Testar em datasets independentes
   - Validação cruzada estratificada
   - Avaliação por especialistas

3. **Explicabilidade**
   - Implementar Grad-CAM
   - Visualizar ativações
   - Documentar decisões

4. **Monitoramento Contínuo**
   - Rastrear performance em produção
   - Detectar drift de dados
   - Atualizar modelo regularmente

5. **Governança**
   - Revisão clínica obrigatória
   - Auditoria de decisões
   - Transparência com pacientes

## 9. Salvar Modelos

In [ ]:
# Salvar modelos
model_scratch.save('cnn_ecg_scratch.keras')
model_transfer.save('cnn_ecg_transfer.keras')

print("✓ Modelos salvos com sucesso!")
print("  - cnn_ecg_scratch.keras")
print("  - cnn_ecg_transfer.keras")

# Download dos modelos (Colab)
try:
    files.download('cnn_ecg_scratch.keras')
    files.download('cnn_ecg_transfer.keras')
    print("\n✓ Download iniciado!")
except:
    print("\n⚠ Execute em ambiente Colab para download automático")

## 10. Conclusões

### Resultados Obtidos

- ✅ CNN do zero implementada e treinada
- ✅ Transfer Learning com MobileNetV2 aplicado
- ✅ Métricas completas calculadas
- ✅ Análise de viés e governança realizada
- ✅ Sistema de inferência funcional

### Limitações

1. **Dataset limitado**: Não representa toda diversidade clínica
2. **Ausência de validação médica**: Não testado por cardiologistas
3. **Contexto acadêmico**: Não adequado para uso clínico real
4. **Possível overfitting**: Necessita validação externa
5. **Falta de explicabilidade**: Decisões não são interpretáveis

### Próximos Passos

1. Implementar Grad-CAM para explicabilidade
2. Testar em datasets externos
3. Aplicar técnicas de balanceamento
4. Realizar fine-tuning dos modelos
5. Validar com especialistas médicos

### ⚠️ AVISO FINAL

**Este projeto é EXCLUSIVAMENTE acadêmico e educacional.**

**NÃO deve ser usado para:**
- Diagnóstico médico real
- Decisões clínicas
- Substituir avaliação profissional

**Sempre consulte profissionais de saúde qualificados para questões médicas.**

---

**CardioAI - FIAP 2026**

**Integrantes:**
- João Vitor Severo Oliveira — RM5666251
- Jonas Luis da Silva — RM561465
- Edson Henrique Felix Batista — RM566321